# Lab 2 - Transfer Learning with the Hugging Face Trainer

**Day 1 - Neural Networks and Deep Learning** | Estimated time: **45 minutes** | CPU-friendly (a GPU is used automatically if present)

**Transfer learning** = reuse a model that already *understands language* and teach it a **new task** with a small amount of labelled data.

You will take **DistilBERT**, pre-trained on Wikipedia and books but never shown a sentiment label, and turn it into a movie-review sentiment classifier.

| Step | What you do | Expected accuracy |
|---|---|---|
| 1 | Measure the untrained model | ~50% (a coin flip) |
| 2 | **Feature extraction**: freeze the body, train only the head | ~80% |
| 3 | **Full fine-tuning**: train every weight | ~85% |
| 4 | Use your model and probe its limits | - |

> **Run the cells in order** with **Shift+Enter**. Training cells take 2-4 minutes on a CPU.

In [ ]:
import os, time, json
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Load optional settings (e.g. HF_TOKEN) from the repository's .env file
from pathlib import Path
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents][:4]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

import numpy as np
import torch
import transformers
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

print("transformers", transformers.__version__, "| torch", torch.__version__)
DEVICE = f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else f"CPU ({os.cpu_count()} cores)"
print("Training will run on:", DEVICE)

## Configuration

Everything you might want to change lives in this one cell.

In [ ]:
BASE_MODEL = "distilbert/distilbert-base-uncased"   # 66M parameters, 6 transformer layers
DATASET    = "stanfordnlp/sst2"                     # Stanford Sentiment Treebank: short movie-review sentences
LABELS     = {0: "NEGATIVE", 1: "POSITIVE"}

TRAIN_SIZE = 2000     # sentences used for training (67,349 available)
MAX_LENGTH = 64       # SST-2 sentences are short
SEED       = 42

# GPU available? You can afford more data:
if torch.cuda.is_available():
    TRAIN_SIZE = 8000

results = {}          # we collect every run here to compare at the end

---
## Step 1 - Load the data

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET)
print(raw)
for row in raw["train"].select(range(5)):
    print(f"  {LABELS[row['label']]:<9} {row['sentence']}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    # No padding here: the data collator pads each batch to its own longest sentence
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LENGTH)

train_ds = (raw["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
            .map(tokenize, batched=True, remove_columns=["idx", "sentence"]))
eval_ds = raw["validation"].map(tokenize, batched=True, remove_columns=["idx", "sentence"])

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print("A tokenized example:", train_ds[0])

SST-2's `test` split has no public labels, so we evaluate on the 872-sentence `validation` split.

---
## Step 2 - Anatomy of a transfer-learning model

`AutoModelForSequenceClassification` = **pre-trained body** + **new classification head**.

In [ ]:
from transformers import AutoModelForSequenceClassification

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2, id2label=LABELS, label2id={v: k for k, v in LABELS.items()})

model = new_model()

body = sum(p.numel() for p in model.distilbert.parameters())
head = sum(p.numel() for n, p in model.named_parameters() if not n.startswith("distilbert."))
print(f"Pre-trained body : {body:>11,} parameters  ({len(model.distilbert.transformer.layer)} transformer layers)")
print(f"New head         : {head:>11,} parameters  (randomly initialised!)")
print()
for name, module in model.named_children():
    if name != "distilbert":
        print(f"  {name}: {module}")

The body turns a sentence into a 768-number vector that captures its meaning.
The head maps that vector to 2 scores (NEGATIVE / POSITIVE). Right now the head is **random**.

---
## Step 3 - The Trainer interface

The `Trainer` runs the whole training loop for you: batching, forward pass, loss, backpropagation,
optimizer and scheduler steps, evaluation, logging, checkpoints, and mixed precision on GPUs.

You give it **four things**:

1. a `model`
2. `TrainingArguments` (the knobs)
3. datasets
4. optionally, a `compute_metrics` function

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

def make_trainer(model, run_name, epochs=1, lr=5e-5, batch_size=16):
    args = TrainingArguments(
        output_dir=f"outputs/{run_name}",
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        weight_decay=0.01,
        warmup_steps=0.1,                 # ramp the learning rate up over the first 10% of steps
        lr_scheduler_type="linear",       # then decay it linearly to 0
        eval_strategy="epoch",            # evaluate after every epoch
        save_strategy="no",               # we save manually at the end
        logging_steps=10,                 # log the loss every 10 steps
        report_to="none",
        fp16=torch.cuda.is_available(),   # mixed precision only on GPU
        dataloader_num_workers=0,         # safest on Windows
        dataloader_pin_memory=torch.cuda.is_available(),
        seed=SEED,
    )
    return Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                   processing_class=tokenizer,
                   data_collator=DataCollatorWithPadding(tokenizer),
                   compute_metrics=compute_metrics)

### 3a. Baseline: evaluate the untrained model

In [ ]:
set_seed(SEED)
trainer = make_trainer(new_model(), "baseline")
m = trainer.evaluate()
results["baseline (untrained)"] = {"accuracy": m["eval_accuracy"], "f1": m["eval_f1"], "train_s": 0, "trainable": 0}
print(f"Untrained accuracy: {m['eval_accuracy']:.2%}  <- the random head is guessing")

### 3b. Strategy 1 - Feature extraction (frozen body)

Freeze every parameter of the pre-trained body: `requires_grad = False`.
Only the ~0.6M head parameters learn. It's fast and needs little data, and the head can use a **large** learning rate.

In [ ]:
set_seed(SEED)
model = new_model()
for p in model.distilbert.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} of {total:,} ({trainable/total:.2%})\n")

trainer = make_trainer(model, "frozen", epochs=2, lr=1e-3)
t0 = time.time()
trainer.train()
secs = time.time() - t0
m = trainer.evaluate()
results["frozen body"] = {"accuracy": m["eval_accuracy"], "f1": m["eval_f1"], "train_s": round(secs), "trainable": trainable}
print(f"\nFrozen-body accuracy: {m['eval_accuracy']:.2%} in {secs:.0f}s")

**Read the training log:** `loss` should trend down, and `eval_accuracy` is reported after each epoch.

### 3c. Strategy 2 - Full fine-tuning

Now **every** weight is updated, so the body itself adapts to sentiment.
Use a **small** learning rate (5e-5) so you nudge the pre-trained knowledge instead of destroying it
(*catastrophic forgetting*).

In [ ]:
set_seed(SEED)
model = new_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

trainer = make_trainer(model, "full", epochs=1, lr=5e-5)
t0 = time.time()
trainer.train()
secs = time.time() - t0
m = trainer.evaluate()
results["full fine-tune"] = {"accuracy": m["eval_accuracy"], "f1": m["eval_f1"], "train_s": round(secs), "trainable": trainable}
print(f"\nFull fine-tune accuracy: {m['eval_accuracy']:.2%} in {secs:.0f}s")

trainer.save_model("outputs/full/final")       # weights + config + tokenizer
print("Saved to outputs/full/final")

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T
df["accuracy"] = df["accuracy"].map("{:.2%}".format)
df["f1"] = df["f1"].astype(float).round(3)
df["trainable"] = df["trainable"].astype(int).map("{:,}".format)
df

**Discuss**

* The model had **never** seen a sentiment label, yet 2,000 examples and a couple of minutes got it to roughly 85%.
  That's the power of pre-training.
* Freezing trains about 1% of the parameters and gets most of the way. When would you choose it?
  (Very little data, weak hardware, or many tasks sharing one body.)
* Full fine-tuning wins on accuracy, but each task needs its own full copy of the model.

---
## Step 4 - Use your model

In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model="outputs/full/final", device=-1)

tests = [
    "An absolute delight from start to finish.",
    "The plot was predictable and the acting was wooden.",
    "I expected to hate it, but it won me over.",
    "Not bad at all.",
    "The visuals are stunning, yet the story is painfully dull.",
    "This app crashes every time I open it.",
]
for t, p in zip(tests, clf(tests)):
    print(f"{p['label']:<9} {p['score']:.3f} | {t}")

**Probe the limits.** Look closely at the predictions above:

* Negation and contrast (*"Not bad at all"*, *"expected to hate it, but..."*) are hard with only 2,000 examples.
* The last sentence is **not a movie review**. The model still answers confidently.
  A classifier only knows the labels it was trained on, a key limit of *predictive* AI compared with *generative* AI.

> **Try it (5 min, pick one)**
> 1. Type your own sentences into the next cell. Can you fool the model?
> 2. Set `TRAIN_SIZE = 6000` in the configuration cell, re-run from Step 1, and train only the full fine-tune. Did the tricky sentences improve? How much longer did it take?
> 3. In the frozen-body cell, try `lr=5e-5` instead of `1e-3`. What happens, and why?

In [ ]:
my_sentences = [
    "Write your own review here",
]
for t, p in zip(my_sentences, clf(my_sentences)):
    print(f"{p['label']:<9} {p['score']:.3f} | {t}")

---
## Summary

* **Transfer learning:** a pre-trained body plus a new task head, trained on a small labelled dataset.
* **Feature extraction** (frozen body) is cheap and fast. **Full fine-tuning** is more accurate.
* The **Trainer** wraps the training loop: `TrainingArguments` + model + data + metrics, then `train()`, `evaluate()` and `save_model()`.
* Learning rate matters: large for a fresh head, small when fine-tuning pre-trained weights.

*Optional:* the same lab exists as plain Python scripts in this folder (`01_baseline.py`, `02_finetune.py`, `03_compare_and_predict.py`) if you prefer the terminal.